In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
 
import re
import os
import time
import subprocess
import tkinter as tk
from tkinter import messagebox
from PIL import Image, ImageTk
 
# 设置ADB路径（根据你的系统和安装路径进行调整）
ADB_PATH = 'C:/Users/DHY-20210315/AppData/Local/Android/Sdk/platform-tools/adb.exe'
# 设置截屏图片显示比例
scl = 0.7
 
# 创建一个GUI窗口
root = tk.Tk()
root.title("ADB辅助点击助手")
screen_width = root.winfo_screenwidth()
screen_height = root.winfo_screenheight()
# 设置窗口大小
window_width = 900
window_height = 600
x = (screen_width - window_width) // 2
y = (screen_height - window_height) // 2
root.geometry(f"{window_width}x{window_height}+{x}+{y}")
 
 
# 函数：通过ADB截屏并显示
def capture_and_display():
    if r_var.get() == '':
        messagebox.showinfo(title='提示', message='没有连接设备呀！')
        return
    # file = str(round(time.time() * 1000)) + '.png'
    file = 'screencap.png'
    scp = '/sdcard/Pictures/' + file
    capture_process = subprocess.Popen([ADB_PATH, '-s', r_var.get(), 'shell', 'screencap', scp])
    capture_process.wait()
 
    lsc = './' + file
    pull_process = subprocess.Popen([ADB_PATH, '-s', r_var.get(), 'pull', scp, lsc])
    pull_process.wait()
 
    screenshot = Image.open(lsc)
    width, height = screenshot.size
    new_width = int(width * scl)
    new_height = int(height * scl)
    screenshot = screenshot.resize((new_width, new_height), Image.ANTIALIAS)
    s_w = new_width + 20
    s_h = new_height + 50
    root.geometry(f"{s_w}x{s_h}+{(screen_width - s_w) // 2}+{(screen_height - s_h) // 2}")
 
    img = ImageTk.PhotoImage(screenshot)
    img_label.config(image=img)
    img_label.image = img
 
 
# 函数：通过ADB点击图片
def click_img(event):
    if r_var.get() == '':
        messagebox.showinfo(title='提示', message='没有连接设备呀！')
        return
    if ck_var.get() == 1:
        for i in range(8):
            subprocess.Popen(
                [ADB_PATH, '-s', r_var.get(), 'shell', 'input', 'tap', str(event.x / scl), str(event.y / scl)])
            time.sleep(0.01)
        ck_var.set(0)
    else:
        subprocess.Popen([ADB_PATH, '-s', r_var.get(), 'shell', 'input', 'tap', str(event.x / scl), str(event.y / scl)])
    time.sleep(1)
    capture_and_display()
 
 
# 函数：通过ADB发送按键
def send_back_command():
    if r_var.get() == '':
        messagebox.showinfo(title='提示', message='没有连接设备呀！')
        return
    subprocess.Popen([ADB_PATH, '-s', r_var.get(), 'shell', 'input', 'keyevent', '4'])
    time.sleep(1)
    capture_and_display()
 
 
# 函数：通过ADB发送滑动
def send_slide_command(arg):
    if r_var.get() == '':
        messagebox.showinfo(title='提示', message='没有连接设备呀！')
        return
    if arg == 1:
        subprocess.Popen([ADB_PATH, '-s', r_var.get(), 'shell', 'input', 'swipe', '969 1050 969 800 100'])
        time.sleep(1)
        capture_and_display()
    else:
        subprocess.Popen([ADB_PATH, '-s', r_var.get(), 'shell', 'input', 'swipe', '969 800 969 1050 100'])
        time.sleep(1)
        capture_and_display()
 
 
# 函数：通过ADB获取设备名
def find_device():
    dvs = os.popen("adb devices").readlines()
    dfs = ''
    for ss in dvs:
        ss = ss.strip('\n')
        if 'List of devices' not in ss and len(ss) > 6 and 'offline' not in ss:
            dv = ss.split('\t')[0]
            p = subprocess.Popen("adb -s %s shell getprop ro.product.model" % dv, stdout=subprocess.PIPE)
            result = p.communicate()
            dn = result[0].decode('utf-8').strip()
            cold_bev = tk.Radiobutton(button_frame, text=dn, variable=r_var, value=dv)
            cold_bev.pack(side="left")
            if dfs == '':
                dfs = dv
    if dfs != '':
        r_var.set(dfs)
 
 
def find_ip(input_string):
    ip_pattern = r'\b(?:[0-9]{1,3}\.){3}[0-9]{1,3}\b'
    ip_addresses = re.findall(ip_pattern, input_string)
    return ip_addresses[0]
 
 
# 函数：通过ADB wifi连接设备
def wifi_connect():
    if r_var.get() == '':
        messagebox.showinfo(title='提示', message='没有连接设备呀！')
        return
    dv = r_var.get()
    if r_var.get().startswith('192.168.'):
        messagebox.showinfo(title='提示', message='已经是WiFi连接了啊！')
        return
    p = subprocess.Popen("adb -s %s shell ip -f inet addr show wlan0" % dv, stdout=subprocess.PIPE)
    result = p.communicate()
    dn = result[0].decode('utf-8').strip()
    ip = find_ip(dn)
    subprocess.Popen([ADB_PATH, 'connect', ip])
 
 
button_frame = tk.Frame(root)
button_frame.pack()
 
capture_button = tk.Button(button_frame, text="截屏", command=capture_and_display)
capture_button.pack(side="left")
tk.Label(button_frame, text="   ").pack(side="left")
back_button = tk.Button(button_frame, text="后退", command=send_back_command)
back_button.pack(side="left")
tk.Label(button_frame, text="   ").pack(side="left")
up_button = tk.Button(button_frame, text="上滑", command=lambda: send_slide_command(1))
up_button.pack(side="left")
tk.Label(button_frame, text="   ").pack(side="left")
down_button = tk.Button(button_frame, text="下滑", command=lambda: send_slide_command(0))
down_button.pack(side="left")
tk.Label(button_frame, text="   ").pack(side="left")
ck_var = tk.IntVar()
c1 = tk.Checkbutton(button_frame, text='8连击', variable=ck_var, onvalue=1, offvalue=0)
c1.pack(side="left")
tk.Label(button_frame, text="   ").pack(side="left")
r_var = tk.StringVar(value='')
find_device()
img_label = tk.Label(root)
img_label.pack()
img_label.bind('<Button-1>', click_img)
wifi_button = tk.Button(button_frame, text="WiFi连接", command=wifi_connect)
wifi_button.pack(side="left")
 
root.mainloop()